In [ ]:
csv_path = '/workspaces/Phan-Tich-va-Quan-Li-Dau-Tu-Nang-Cao/output/historical_stock_data_full.csv'

In [ ]:
!pip install scipy

In [ ]:
# Khai báo thư viện cho toàn bộ tài liệu

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import matplotlib.ticker as mtick
import numpy as np
from scipy.optimize import minimize

In [ ]:
# Kiểm tra số liệu

df_week2 = pd.read_csv(csv_path)
print("5 hàng đầu tiên của DataFrame:")
display(df_week2.head())

In [ ]:
# Lọc danh mục cổ phiếu theo mã để phân tích
selected_symbols = ['MHC', 'MBB', 'FRT', 'HAH', 'VGI', 'KDH']

symbol_col = next((c for c in df_week2.columns if c.lower() in ['symbol', 'ticker', 'code', 'stock']), None)

df_portfolio = df_week2[df_week2[symbol_col].isin(selected_symbols)].copy()
print('Số lượng bản ghi trong danh mục:', len(df_portfolio))
print('Các mã xuất hiện trong danh mục:')
print(sorted(df_portfolio[symbol_col].unique()))

display(df_portfolio.head())

In [ ]:
# Tính lợi nhuận kỳ vọng (weighted average) cho từng tài sản
df_portfolio = df_portfolio.sort_values([symbol_col, 'Date']).copy()
df_portfolio['Return'] = df_portfolio.groupby(symbol_col)['Close'].pct_change()

expected_returns = (
    df_portfolio.dropna(subset=['Return'])
    .groupby(symbol_col)['Return']
    .apply(lambda x: np.average(x, weights=np.ones(len(x))))
    .rename('Expected_Return')
)

expected_returns_pct = expected_returns * 100
print('Lợi nhuận kỳ vọng của từng tài sản (weighted average trên các phiên giao dịch):')
display(expected_returns_pct.round(4).rename('Expected_Return_%'))

# Tính phương sai và hiệp phương sai cho lợi nhuận
returns_pivot = (
    df_portfolio[['Date', symbol_col, 'Return']]
    .dropna(subset=['Return'])
    .groupby(['Date', symbol_col])['Return']
    .mean()
    .unstack(symbol_col)
)

variance = returns_pivot.var(ddof=1).rename('Variance')
covariance = returns_pivot.cov(ddof=1)

print('Phương sai của lợi nhuận từng tài sản:')
display(variance.round(8))
print('Ma trận hiệp phương sai giữa các tài sản:')
display(covariance.round(8))

# Vẽ Biên giới hiệu quả (Efficient Frontier)?

Sau khi chạy tối ưu hóa cho nhiều mức lợi nhuận khác nhau, chúng ta thu được một tập hợp các danh mục tối ưu. 
Đường cong này là Biên giới hiệu quả: tại mỗi mức lợi nhuận, nó cho thấy danh mục có rủi ro nhỏ nhất.
Các danh mục nằm bên trong vùng khả thi nhưng không nằm trên biên giới sẽ bị áp đảo bởi các danh mục khác có lợi nhuận cao hơn hoặc rủi ro thấp hơn.


In [ ]:
from scipy.optimize import minimize
# Tối ưu hóa danh mục trên Biên giới hiệu quả (Efficient Frontier)
returns_matrix = returns_pivot.dropna(axis=1, how='all')
cov_matrix = returns_matrix.cov(ddof=1).values
mean_returns = expected_returns.reindex(returns_matrix.columns).values

n_assets = len(returns_matrix.columns)
bounds = tuple((0, 1) for _ in range(n_assets))

def portfolio_variance(weights):
    return weights.T @ cov_matrix @ weights

def portfolio_return(weights):
    return weights @ mean_returns

def weight_sum_constraint(weights):
    return np.sum(weights) - 1

def target_return_constraint_factory(target_return):
    return {'type': 'eq', 'fun': lambda w: portfolio_return(w) - target_return}

initial_weights = np.repeat(1 / n_assets, n_assets)

target_returns = np.linspace(mean_returns.min(), mean_returns.max(), 20)
efficient_weights = []
efficient_variances = []
efficient_returns = []

for target in target_returns:
    constraints = [
        {'type': 'eq', 'fun': weight_sum_constraint},
        target_return_constraint_factory(target),
    ]
    result = minimize(
        portfolio_variance,
        initial_weights,
        method='SLSQP',
        bounds=bounds,
        constraints=constraints,
        options={'ftol': 1e-12, 'disp': False},
    )
    if result.success:
        efficient_weights.append(result.x)
        efficient_variances.append(result.fun)
        efficient_returns.append(portfolio_return(result.x))
    else:
        efficient_weights.append(np.full(n_assets, np.nan))
        efficient_variances.append(np.nan)
        efficient_returns.append(np.nan)

efficient_frontier = pd.DataFrame({
    'Target_Return': np.array(efficient_returns),
    'Variance': np.array(efficient_variances),
    'Std_Dev': np.sqrt(np.array(efficient_variances)),
}, index=[f'{r:.4f}' for r in target_returns])

allocation_df = pd.DataFrame(efficient_weights, columns=returns_matrix.columns)
allocation_df.index = [f'{r:.4f}' for r in efficient_returns]



In [ ]:
print('Biên giới hiệu quả (Efficient Frontier):')
display(efficient_frontier.round(8))

print('Một số tỷ trọng danh mục tối ưu cho mức lợi nhuận mục tiêu:')
display(allocation_df.head(10).round(4))

plt.figure(figsize=(8, 5))
plt.plot(efficient_frontier['Std_Dev'], efficient_frontier['Target_Return'], marker='o')
plt.title('Efficient Frontier')
plt.xlabel('Rủi ro danh mục (Độ lệch chuẩn)')
plt.ylabel('Lợi nhuận kỳ vọng danh mục')
plt.grid(True)
plt.show()

# Danh mục Tangency và Đường thị trường vốn (CML)

Khi thêm tài sản phi rủi ro vào phân tích, ta có thể xác định danh mục tiếp điểm (Tangency Portfolio): 
danh mục có tỷ số Sharpe lớn nhất, tức là lợi nhuận vượt trội cao nhất trên mỗi đơn vị rủi ro.

Đường thị trường vốn (Capital Market Line - CML) là đường thẳng bắt đầu từ lãi suất phi rủi ro và tiếp xúc với Biên giới hiệu quả tại điểm Tangency.


In [ ]:
# Xác định Tangency Portfolio và vẽ CML
risk_free_rate = 0.0001  # Giá trị giả định lãi suất phi rủi ro theo cùng đơn vị (ngày)

def negative_sharpe(weights):
    port_return = portfolio_return(weights)
    port_vol = np.sqrt(portfolio_variance(weights))
    return - (port_return - risk_free_rate) / port_vol

constraints = [{'type': 'eq', 'fun': weight_sum_constraint}]
result_tangency = minimize(
    negative_sharpe,
    initial_weights,
    method='SLSQP',
    bounds=bounds,
    constraints=constraints,
    options={'ftol': 1e-12, 'disp': False},
)

if not result_tangency.success:
    raise RuntimeError('Không tìm được danh mục Tangency:', result_tangency.message)

tangency_weights = result_tangency.x
tangency_return = portfolio_return(tangency_weights)
tangency_vol = np.sqrt(portfolio_variance(tangency_weights))
tangency_sharpe = (tangency_return - risk_free_rate) / tangency_vol

tangency_df = pd.DataFrame({
    'Weight': tangency_weights,
    'Asset': returns_matrix.columns
}).set_index('Asset')

print('Tangency Portfolio:')
print(f' - Lợi nhuận kỳ vọng: {tangency_return:.8f}')
print(f' - Độ lệch chuẩn: {tangency_vol:.8f}')
print(f' - Sharpe Ratio: {tangency_sharpe:.8f}')
display(tangency_df.round(4))

# Vẽ CML
cml_x = np.linspace(0, tangency_vol * 1.2, 100)
cml_y = risk_free_rate + tangency_sharpe * cml_x
plt.figure(figsize=(8, 5))
plt.plot(efficient_frontier['Std_Dev'], efficient_frontier['Target_Return'], marker='o', label='Efficient Frontier')
plt.plot(cml_x, cml_y, label='Capital Market Line (CML)', linestyle='--')
plt.scatter([tangency_vol], [tangency_return], color='red', label='Tangency Portfolio', zorder=5)
plt.scatter([0], [risk_free_rate], color='green', label='Risk-free rate', zorder=5)
plt.title('Efficient Frontier và Capital Market Line')
plt.xlabel('Rủi ro danh mục (Độ lệch chuẩn)')
plt.ylabel('Lợi nhuận kỳ vọng danh mục')
plt.legend()
plt.grid(True)
plt.show()

# Danh mục Tangency và Đường thị trường vốn (CML)Khi thêm tài sản phi rủi ro vào phân tích, ta có thể xác định danh mục tiếp điểm (Tangency Portfolio):danh mục có tỷ số Sharpe lớn nhất, tức là lợi nhuận vượt trội cao nhất trên mỗi đơn vị rủi ro.Đường thị trường vốn (Capital Market Line - CML) là đường thẳng bắt đầu từ lãi suất phi rủi ro và tiếp xúc với Biên giới hiệu quả tại điểm Tangency.# Bước 5: Phân bổ tỷ trọng tài sản (Asset Allocation)Tại điểm có tỷ số Sharpe tối đa, mô hình cho biết chính xác tỷ lệ phần trăm vốn cần đầu tư vào từng cổ phiếu. Ở danh mục này, các rủi ro riêng biệt đã được đa dạng hóa hoàn toàn, chỉ còn lại rủi ro hệ thống.

In [ ]:
print(df_portfolio.columns)

In [ ]:
import numpy as np
import pandas as pd
from scipy.optimize import minimize

# Định nghĩa lại các giới hạn trần/sàn tỷ trọng
max_weight_limit = 0.25  # Trần 25% cho mỗi cổ phiếu
min_weight_limit = 0.05  # Sàn 5% để không mã nào bị loại bỏ về 0%

# Tự động lấy số lượng tài sản từ ma trận returns_matrix của bạn để tránh lỗi NameError
num_assets = len(returns_matrix.columns)

# Tạo ràng buộc mới
bounds_constrained = tuple((min_weight_limit, max_weight_limit) for _ in range(num_assets))

# Chạy lại tối ưu hóa SLSQP với ràng buộc an toàn mới
result_tangency_safe = minimize(
    daily_negative_sharpe,
    initial_weights,
    method='SLSQP',
    bounds=bounds_constrained,  # Áp dụng giới hạn an toàn mới
    constraints=constraints,
    options={'ftol': 1e-12, 'disp': False}
)

if not result_tangency_safe.success:
    raise RuntimeError('Không tìm được danh mục tối ưu an toàn:', result_tangency_safe.message)

# Xuất bảng tỷ trọng thực chiến
tangency_weights_safe = result_tangency_safe.x
allocation_safe_df = pd.DataFrame({
    'Cổ Phiếu': returns_matrix.columns,
    'Tỷ Trọng Thực Chiến (%)': np.round(tangency_weights_safe * 100, 2)
}).sort_values(by='Tỷ Trọng Thực Chiến (%)', ascending=False)

print("=" * 60)
print("TỶ TRỌNG PHÂN BỔ VỐN THỰC CHIẾN (ĐÃ ÁP TRẦN RỦI RO 25%)")
print("=" * 60)
display(allocation_safe_df)

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.optimize import minimize

# ==========================================
# 1. THIẾT LẬP THAM SỐ VÀ CẤU HÌNH AN TOÀN
# ==========================================
# Giả định lãi suất phi rủi ro theo ngày (0.01%/ngày)
risk_free_rate = 0.0001  

# Khởi tạo số lượng tài sản và tỷ trọng ban đầu (chia đều)
num_assets = len(returns_matrix.columns)
initial_weights = np.array(num_assets * [1. / num_assets])

# Định nghĩa các giới hạn trần/sàn thực chiến để danh mục không bị cực đoan
max_weight_limit = 0.25  # Trần 25% cho mỗi cổ phiếu
min_weight_limit = 0.05  # Sàn 5% để không mã nào bị loại bỏ về 0%
bounds_constrained = tuple((min_weight_limit, max_weight_limit) for _ in range(num_assets))

# ==========================================
# 2. ĐỊNH NGHĨA CÁC HÀM MỤC TIÊU (ĐƠN VỊ NGÀY)
# ==========================================
# Ràng buộc: Tổng tỷ trọng vốn phải bằng 100% (sum(w) = 1)
def weight_sum_constraint(weights):
    return np.sum(weights) - 1

# Hàm mục tiêu: Cực tiểu hóa trị âm của Sharpe (tương đương cực đại hóa Sharpe)
def daily_negative_sharpe(weights):
    # Sử dụng các hàm portfolio_return và portfolio_variance sẵn có của bạn
    port_return = portfolio_return(weights)
    port_vol = np.sqrt(portfolio_variance(weights))
    return - (port_return - risk_free_rate) / port_vol

# ==========================================
# 3. TIẾN HÀNH TỐI ƯU HÓA BẰNG SLSQP
# ==========================================
constraints = [{'type': 'eq', 'fun': weight_sum_constraint}]

result_tangency_safe = minimize(
    daily_negative_sharpe,
    initial_weights,
    method='SLSQP',
    bounds=bounds_constrained,  # Áp dụng cấu hình trần/sàn an toàn
    constraints=constraints,
    options={'ftol': 1e-12, 'disp': False}
)

if not result_tangency_safe.success:
    raise RuntimeError('Không tìm được danh mục tối ưu an toàn:', result_tangency_safe.message)

# Lấy kết quả tối ưu
tangency_weights = result_tangency_safe.x
tangency_return = portfolio_return(tangency_weights)
tangency_vol = np.sqrt(portfolio_variance(tangency_weights))
tangency_sharpe = (tangency_return - risk_free_rate) / tangency_vol

# ==========================================
# 4. TRỰC QUAN HÓA KẾT QUẢ PHÂN BỔ VỐN
# ==========================================
tangency_df = pd.DataFrame({
    'Tỷ Trọng Vốn (%)': np.round(tangency_weights * 100, 2),
    'Asset': returns_matrix.columns
}).set_index('Asset').sort_values(by='Tỷ Trọng Vốn (%)', ascending=False)

print("=" * 60)
print("TỶ TRỌNG PHÂN BỔ VỐN THỰC CHIẾN TẠI ĐIỂM TANGENCY PORTFOLIO")
print("=" * 60)
display(tangency_df)

print('\nThông số danh mục Tangency (Daily):')
print(f' - Lợi nhuận kỳ vọng hàng ngày: {tangency_return:.6f}')
print(f' - Độ lệch chuẩn hàng ngày (Rủi ro): {tangency_vol:.6f}')
print(f' - Sharpe Ratio đạt được: {tangency_sharpe:.4f}')

# ==========================================
# 5. VẼ ĐƯỜNG BIÊN HIỆU QUẢ VÀ ĐƯỜNG CML
# ==========================================
# Khởi tạo dữ liệu vẽ đường CML thẳng kéo dài qua điểm tiếp điểm
cml_x = np.linspace(0, tangency_vol * 1.3, 100)
cml_y = risk_free_rate + tangency_sharpe * cml_x

plt.figure(figsize=(10, 6))

# Vẽ đường Biên hiệu quả (Efficient Frontier) từ dữ liệu bạn đã tính trước đó
plt.plot(efficient_frontier['Std_Dev'], efficient_frontier['Target_Return'], 
         marker='o', color='#1f77b4', linewidth=2, label='Efficient Frontier')

# Vẽ đường thị trường vốn Capital Market Line (CML)
plt.plot(cml_x, cml_y, color='#ff7f0e', linestyle='--', linewidth=2, label='Capital Market Line (CML)')

# Chấm điểm danh mục tiếp điểm lý tưởng (Tangency Portfolio)
plt.scatter([tangency_vol], [tangency_return], color='red', s=100, 
            label='Tangency Portfolio (Max Sharpe)', zorder=5)

# Chấm điểm tài sản phi rủi ro (Risk-free rate)
plt.scatter([0], [risk_free_rate], color='green', s=100, 
            label=f'Risk-free rate ({risk_free_rate*100:.2f}%)', zorder=5)

plt.title('TỐI ƯU HÓA DANH MỤC: Efficient Frontier và Capital Market Line (CML)', fontsize=14, fontweight='bold')
plt.xlabel('Rủi ro danh mục (Độ lệch chuẩn - Ngày)', fontsize=12)
plt.ylabel('Lợi nhuận kỳ vọng danh mục (Ngày)', fontsize=12)
plt.legend(loc='upper left', fontsize=11)
plt.grid(True, linestyle=':', alpha=0.6)
plt.tight_layout()
plt.show()

# BÁO CÁO NGHIÊN CỨU VÀ TỐI ƯU HÓA DANH MỤC ĐẦU TƯ ĐỊNH LƯỢNG

## I. Giới Thiệu Hệ Phương Pháp Phân Tích
Nghiên cứu này áp dụng phương pháp tiếp cận định lượng (Quant Approach) dựa trên **Lý thuyết Danh mục đầu tư Hiện đại (Modern Portfolio Theory - MPT)** kết hợp với các bộ lọc quản trị rủi ro cực đoan để xây dựng một danh mục tối ưu, loại bỏ hoàn toàn yếu tố cảm tính. Quy trình gồm nhiều lớp lọc nghiêm ngặt từ dữ liệu lịch sử giai đoạn 2021 - 2026.

---

## II. Phân Tích Sâu Cấu Trúc Rủi Ro Phân Phối

### 1. Lịch Sử Sụt Giảm Tài Sản (Drawdown Curves)
Biểu đồ theo dõi dòng thời gian sụt giảm (Drawdown) vạch trần hành vi chịu đựng thực tế của các tài sản trong các pha khủng hoảng hệ thống (đặc biệt là giai đoạn nửa cuối năm 2022):
* **Đáy vực thẳm chu kỳ (HAH):** Lao dốc dốc đứng xuống vùng quanh $-73.5\%$. Đây là đặc trưng của nhóm ngành vận tải biển hạ tầng khi hết chu kỳ cao điểm, giá cước sập khiến cổ phiếu bị bán tháo không phanh. Tuy nhiên, mã này sở hữu lực bật "leo núi" ngược trở lại rất nhanh.
* **"Tấm khiên" phòng thủ (MBB & MCH):** Có đáy sụt giảm nông hơn thị trường chung rất nhiều và là những mã đầu tiên tìm đường quay trở lại mốc $0\%$ (vùng đỉnh lịch sử cũ).

### 2. Ma Trận Hình Dáng Lợi Nhuận (Skewness vs Kurtosis)
Khảo sát phân phối hình dáng lợi nhuận (tô màu theo thang rủi ro MDD) giúp nhận diện bản chất cấu trúc biến động của từng mã:
* **Vùng "Tử thần" (Skewness < 0 & Kurtosis cao):** Điển hình là `HHV` và `VGI`. Khi thị trường gặp biến cố, nhóm này mất thanh khoản rất nhanh, tạo ra bẫy rủi ro **"Lên thang bộ, xuống thang máy"**, tàn phá tài sản nghiêm trọng.
* **Vùng "Đột biến tích cực" (Skewness > 0 & Kurtosis thấp):** Điển hình là `FRT` và `HAH`. Đây là vùng lý tưởng cho các mã tăng trưởng chu kỳ: tích lũy nền chặt nhưng khi vào sóng thì bùng nổ tăng trần liên tục (**"Lên thang máy, xuống thang bộ"**).
* **Trường hợp cá biệt (MCH):** Sở hữu Kurtosis cao ($~9$) nhưng Skewness dương. Điều này minh chứng cho một dạng biến động **sốc tăng giá** chứ không phải sốc giảm giá gãy nền.

---

## III. Kết Quả Tối Ưu Hóa Tỷ Trọng Vốn Thực Chiến

Khi chạy mô hình tối ưu hóa toán học nguyên bản nhằm đạt chỉ số **Max Sharpe Ratio**, thuật toán có xu hướng hoạt động cực đoan: dồn hơn $82\%$ vốn vào đúng hai mã có lực công mạnh nhất (`FRT` và `HAH`) và bỏ rơi các mã khác về $0\%$.

Để biến mô hình toán học thành một danh mục thực tế có thể giải ngân an toàn, bộ tham số **Ràng buộc trần tỷ trọng (Constrained Bounds)** đã được áp đặt:
* **Mức trần (Max Weight):** $25.00\%$ cho một mã để tránh rủi ro "bỏ trứng vào một giỏ".
* **Mức sàn (Min Weight):** $5.00\%$ để đảm bảo tính đa dạng hóa, không bỏ sót các mã vệ tinh.

### Bảng Phân Bổ Tỷ Trọng Vốn Thực Chiến (Max Sharpe Constrained)

| STT | Mã Cổ Phiếu | Ngành Nghề | Tỷ Trọng Vốn (%) | Vai Trò Chiến Lược |
| :---: | :---: | :--- | :---: | :--- |
| 1 | **FRT** | Bán lẻ / Dược phẩm | **25.00%** | Mũi nhọn tấn công chủ lực |
| 2 | **HAH** | Vận tải biển / Logistics | **25.00%** | Mũi nhọn chu kỳ, tận dụng sức bật giá |
| 3 | **MBB** | Ngân hàng | **25.00%** | Lõi phòng thủ tài chính, kiểm soát xung lực |
| 4 | **MCH** | Hàng tiêu dùng thiết yếu | **8.65%** | Vệ tinh giữ tiền, dòng tiền bất chấp vĩ mô |
| 5 | **VGI** | Viễn thông quốc tế | **8.37%** | Vệ tinh đột phá, đại diện mảng Công nghệ |
| 6 | **KDH** | Bất động sản dân cư | **7.98%** | Đa dạng hóa tương quan ngành |
| | **Tổng cộng** | | **100.00%** | |

### Thông Số Đầu Ra Danh Mục Tangency Portfolio (Daily)
* **Lợi nhuận kỳ vọng hàng ngày:** $0.001627$
* **Độ lệch chuẩn hàng ngày (Rủi ro):** $0.018173$
* **Sharpe Ratio đạt được:** $0.0840$ 
* **Sharpe Ratio quy năm (Annualized Sharpe):** $\approx \mathbf{1.33}$ *(Danh mục đạt trạng thái xuất sắc theo tiêu chuẩn quản lý quỹ quốc tế).*

---

## IV. Đánh Giá Và Lưu Ý Các Hạn Chế Của Mô Hình

Mặc dù danh mục đã được tối ưu hóa bài bản, việc vận hành trong thực tế bắt buộc phải lưu ý 3 giới hạn cốt lộ của Lý thuyết Markowitz:

### 1. Độ nhạy của dữ liệu đầu vào (Input Sensitivity)
Mô hình toán học rất nhạy cảm với tập dữ liệu quá khứ dùng để tính toán. Một thay đổi nhỏ trong việc chọn khung thời gian (Window size) sẽ bẻ gãy dự báo lợi nhuận kỳ vọng và ma trận hiệp phương sai, dẫn đến việc đề xuất một tỷ trọng hoàn toàn khác. Do đó, tỷ trọng này mang tính chất động, không phải tĩnh.

### 2. Giả định về Phân phối chuẩn (Gaussian Distribution)
Mô hình MPT giả định lợi nhuận cổ phiếu di chuyển theo hình chuông đối xứng. Trên thực tế, thị trường chứng khoán Việt Nam thường xuyên xuất hiện hiện tượng **Kurtosis cao (đuôi dày)** và **Skewness lệch**. Các sự kiện sập gãy hệ thống hoặc tăng nóng bất ngờ xảy ra với tần suất dày hơn mô hình tính toán. Việc kết hợp chỉ số sụt giảm tài sản (MDD) thực tế vào bộ lọc trước đó là bắt buộc để sửa sai cho giả định này.

### 3. Rủi ro Quá khớp (Overfitting / Looking-Back Bias)
Danh mục tối ưu Max Sharpe thực chất là một danh mục "nhìn lại quá khứ" để tối ưu hóa. Nếu bối cảnh vĩ mô hoặc chu kỳ ngành trong tương lai thay đổi hoàn toàn, danh mục sẽ không đạt được hiệu suất như kỳ vọng.

---

## V. Chiến Thuật Hành Động Thực Chiến

1. **Cơ chế triệt tiêu rủi ro:** Tại điểm tiếp điểm Tangency Portfolio, các rủi ro riêng biệt (phi hệ thống) của từng doanh nghiệp đã được đa dạng hóa hoàn toàn nhờ ma trận bù trừ hiệp biến. Danh mục lúc này chỉ còn đối mặt với rủi ro hệ thống (rủi ro toàn thị trường).
2. **Nguyên tắc đi tiền:** Giải ngân chuẩn theo tỷ trọng thực chiến. Định kỳ hàng quý hoặc bán niên cần tiến hành **Tái cân bằng danh mục (Rebalancing)** để kéo tỷ trọng các mã tăng quá nóng hoặc giảm quá sâu về lại bộ khung chuẩn ban đầu.
3. **Tâm lý quản trị:** Khi rủi ro hệ thống xảy ra khiến toàn bộ thị trường lao dốc, các đường drawdown tự động bị kéo xuống, đó là hiện tượng bình thường của thị trường. Đây không phải lỗi chọn mã, mà là cơ hội kích hoạt dòng tiền bên ngoài để gia tăng tỷ trọng chính 6 mã chiến lược này với giá chiết khấu.